<div style="background-color:#000047; padding: 30px; border-radius: 10px; color: white; text-align: center;">
    <img src='Figures/alinco_white_text.png' style="height: 100px; margin-bottom: 10px;"/>
    <h1>Módulo 1: Preprocesamiento y Representación de Texto</h1>
    <h2>Tokenización, Stemming y Lematización</h2>
</div>


Tokenización, Stemming y Lematización son las tres técnicas fundamentales de preprocesamiento morfológico en NLP. Se aplican en secuencia para normalizar el texto antes de cualquier tarea posterior.

## 1. Tokenización 

### ¿Qué es la tokenización?

La **tokenización** es el proceso de dividir un texto continuo en unidades llamadas **tokens**. Es el **primer paso obligatorio** en cualquier pipeline de NLP.

```
"El NLP es fascinante."
        │
        ▼  tokenización
["El", "NLP", "es", "fascinante", "."]
```

### Tipos de tokenización

| Tipo | Descripción | Ejemplo |
|---|---|---|
| **Por palabras** | Divide en palabras y signos | `["Hola", ",", "mundo", "!"]` |
| **Por oraciones** | Divide en oraciones | `["Hola.", "¿Cómo estás?"]` |
| **Por caracteres** | Un token = un carácter | `['H','o','l','a']` |
| **Subword (BPE)** | Divide palabras raras en piezas | `"unhappy"` → `["un", "##happy"]` |

### Retos de la tokenización

- **Contracciones:** `don't` → `["do", "n't"]` o `["don't"]`?
- **Palabras compuestas:** `Nueva York`, `machine learning`
- **Puntuación ambigua:** `Dr. García` vs `fin de oración.`
- **URLs y correos:** `usuario@mail.com` no debe dividirse
- **Idiomas sin espacios:** chino, japonés, tailandés

### Tokenización (sin librerías)

In [ ]:
import re

texto = "El Dr. García llegó a las 8:30 a.m. ¡Qué sorpresa! NLP es fascinante."


In [ ]:
#  Método 1: split() básico (naive) 
tokens_naive = texto.split()
print("Método 1  split() básico:")
print(f"  {tokens_naive}")
print(f"  Tokens: {len(tokens_naive)}")


In [ ]:
#  Método 2: regex por palabras 
tokens_regex = re.findall(r"\b\w+\b", texto)
print("Método 2  regex \\b\\w+\\b:")
print(f"  {tokens_regex}")


In [ ]:
#  Método 3: regex con puntuación separada 
tokens_punct = re.findall(r"\w+(?:['-]\w+)*|[.,!?;:¡¿]", texto)
print("Método 3  regex con puntuación:")
print(f"  {tokens_punct}")

In [ ]:
# Tokenizador de oraciones desde cero
def tokenizar_oraciones(texto):
    """Divide texto en oraciones usando regex."""
    patron = r'(?<=[.!?¿¡])\s+(?=[A-ZÁÉÍÓÚ])|(?<=[.!?])\s*$'
    oraciones = re.split(r'(?<=[.!?])\s+', texto.strip())
    return [s.strip() for s in oraciones if s.strip()]



In [ ]:
parrafo = """El procesamiento de lenguaje natural es emocionante. 
¿Alguna vez te preguntaste cómo funciona Google Translate? 
Hoy aprenderemos las bases. ¡Empecemos!"""

oraciones = tokenizar_oraciones(parrafo)


In [ ]:
oraciones

### Tokenización con NLTK

NLTK ofrece múltiples tokenizadores para distintos propósitos:

| Tokenizador | Función | Caso de uso |
|---|---|---|
| `word_tokenize` | Palabras + puntuación (TreebankWordTokenizer) | Texto general |
| `sent_tokenize` | Oraciones (Punkt) | División de oraciones |
| `TweetTokenizer` | Maneja emojis y @menciones | Redes sociales |
| `MWETokenizer` | Multi-word expressions | "Nueva York" como un token |
| `RegexpTokenizer` | Patrón regex personalizado | Casos especiales |

In [ ]:
texto = """El aprendizaje automático y el deep learning son áreas clave del NLP. 
¡Los modelos como BERT y GPT-4 han revolucionado el campo! 
Visita: https://huggingface.co para más info."""

In [ ]:
import nltk
from nltk.tokenize import (
    word_tokenize, sent_tokenize,
    TweetTokenizer, MWETokenizer, RegexpTokenizer
)

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

In [ ]:
#  word_tokenize
word_tokenize(texto)

In [ ]:
#  sent_tokenize 
sent_tokenize(texto)

In [ ]:
# TweetTokenizer 
tweet = "¡Me encanta el #NLP! 😍🚀 @OpenAI sigue innovando... https://t.co/xyz"


In [ ]:
tokenizer = TweetTokenizer()
tweet_tokenizer = tokenizer.tokenize(tweet)
tweet_tokenizer

In [ ]:
#  RegexpTokenizer 
rt = RegexpTokenizer(r'[a-záéíóúüñ]+')
texto_simple = "Hola, ¿cómo estás? Bien, gracias."
rt

In [ ]:
help(rt)

In [ ]:
rt.tokenize(texto_simple)

### Tokenización con spaCy 

spaCy tokeniza de forma más sofisticada: separa contracciones, maneja casos especiales y aplica reglas por idioma.

In [ ]:
#Importar Spacy
import spacy

try:
    nlp_es = spacy.load("es_core_news_sm")
    nlp_en = spacy.load("en_core_web_sm")
except OSError:
    import subprocess
    subprocess.run(["python", "-m", "spacy", "download", "es_core_news_sm"])
    subprocess.run(["python", "-m", "spacy", "download", "en_core_web_sm"])
    nlp_es = spacy.load("es_core_news_sm")
    nlp_en = spacy.load("en_core_web_sm")


In [ ]:
# Texto en Español e Inglés
texto_es = "El procesamiento del lenguaje natural, o NLP, es fascinante."
texto_en = "I don't think it's necessary, but let's try anyway!"

doc_es = nlp_es(texto_es)
doc_en = nlp_en(texto_en)


In [ ]:
doc_es

In [ ]:
#Tokens
print("spaCy con Tokens en ESPAÑOL:")
print(f"  {[t.text for t in doc_es]}")

print()
print("spaCy con Tokens en INGLÉS (contracciones):")
print(f"  {[t.text for t in doc_en]}")

print()
print("spaCy con Propiedades de cada token (español):")
print(f"  {'Token':<15} {'¿Alfa?':<8} {'¿Puntuación?':<14} {'¿Espacio?':<10} {'Pos inicio'}")
print("  " + "-"*55)

for t in doc_es:
    print(f"  {t.text:<15} {str(t.is_alpha):<8} {str(t.is_punct):<14} {str(t.is_space):<10} {t.idx}")

#### Explorando más caracteríticas de Spacy

In [ ]:
# Import spaCy con inglés
import spacy
nlp = spacy.load('en_core_web_sm')

In [ ]:
# Create a string that includes opening and closing quotation marks
mystring = '"We\'re moving to L.A.!"'
print(mystring)

In [ ]:
# Creando un objeto documento
doc = nlp(mystring)

for token in doc:
    print(token.text, end=' | ')

<img src="Figures/tokenization.png" width="600">

-  **Prefijos**:	Character(s) at the beginning &#9656; `$ ( “ ¿`
-  **Sufijos**:	Character(s) at the end &#9656; `km ) , . ! ”`
-  **Infijo**:	Character(s) in between &#9656; `- -- / ...`
-  **Excepción**: Special-case rule to split a string into several tokens or prevent a token from being split when punctuation rules are applied &#9656; `St. U.S.`

Notar que los tokens son partes del texto original. Los tokens son los bloques de construcción básicos de un objeto Doc: todo lo que ayuda a entender el significado del texto se deriva de los tokens y su relación entre sí.

#### Prefijos, sufijos e infijos

spaCy aislará la puntuación que *no* forma parte integral de una palabra. A las comillas, comas y los puntos finales de una oración se les asignará su propio token. Sin embargo, la puntuación que existe como parte de una dirección de correo electrónico, sitio web o valor numérico se mantendrá como parte del token.

In [ ]:
doc2 = nlp_en(u"We're here to help! Send snail-mail, email support@oursite.com or visit us at http://www.oursite.com!")

for t in doc2:
    print(t)

Tenga en cuenta que los signos de exclamación, la coma y el guión en 'snail-mail' tienen asignados sus propios tokens, pero se conservan tanto la dirección de correo electrónico como el sitio web.

In [ ]:
doc3 = nlp(u'A 5km NYC cab ride costs $10.30')

for t in doc3:
    print(t)

#### Excepciones
La puntuación que existe como parte de una abreviatura conocida se mantendrá como parte del token.

In [ ]:
doc4 = nlp(u"Let's visit St. Louis in the U.S. next year.")

for t in doc4:
    print(t)

Aquí se conservan las abreviaturas de "Saint" y "United States".

#### Contando Tokens
Los objetos `Doc` tienen un número determinado de tokens:

In [ ]:
len(doc)

#### Contar entradas de vocabulario
¡Los objetos `Vocab` contienen una biblioteca completa de elementos!

In [ ]:
len(doc.vocab)

In [ ]:
doc.vocab

**NOTA:** Este número cambia según la biblioteca de idiomas cargada al principio y cualquier lexema nuevo introducido en el `vocab` cuando se creó el` Doc`

#### Recuperación de Tokens

Los objetos `Doc` se pueden considerar como listas de objetos `token`. Como tal, los tokens individuales se pueden recuperar por posición de índice, y los tramos de tokens se pueden recuperar mediante indexación
: 

In [ ]:
doc5 = nlp(u'It is better to give than to receive.')

# Retrieve the third token:
doc5[2]

In [ ]:
# Retrieve three tokens from the middle:
doc5[2:5]

In [ ]:
# Retrieve the last four tokens:
doc5[-4:]

#### Los tokens no se pueden reasignar
Aunque los objetos `Doc` pueden considerarse listas de tokens, *no* admiten la reasignación de elementos:

In [ ]:
doc6 = nlp(u'My dinner was horrible.')
doc7 = nlp(u'Your dinner was delicious.')

In [ ]:
doc6[3]

In [ ]:
doc7[3]

In [ ]:
# Try to change "My dinner was horrible" to "My dinner was delicious"
doc6[3] = doc7[3]

#### Entidades nombradas
Yendo un paso más allá de los tokens, *entidades con nombre* agregan otra capa de contexto. El modelo de lenguaje reconoce que ciertas palabras son nombres de organizaciones mientras que otras son ubicaciones, y otras combinaciones se relacionan con dinero, fechas, etc. Las entidades nombradas son accesibles a través de la propiedad `ents` de un objeto` Doc`.

In [ ]:
doc8 = nlp(u'Apple to build a Hong Kong factory for $6 million')

for token in doc8:
    print(token.text, end=' | ')

print('\n----')

for ent in doc8.ents:
    print(ent.text+' - '+ent.label_+' - '+str(spacy.explain(ent.label_)))

Observe cómo dos tokens se combinan para formar la entidad "Hong Kong" y tres tokens se combinan para formar la entidad monetaria: "$ 6 millones".

In [ ]:
len(doc8.ents)

El reconocimiento de entidades con nombre (NER) es una herramienta importante de aprendizaje automático aplicada al procesamiento del lenguaje natural. <br> Haremos mucho más con esto en siguientes sesiones. Para obtener más información sobre **entidades nombradas**, visite https://spacy.io/usage/linguistic-features#named-entities

#### Noun chunks

Similar a `Doc.ents`,` Doc.noun_chunks` es otra propiedad del objeto. *Noun chunks* corresponde a "frases sustantivas básicas": frases planas que tienen un sustantivo como encabezado. 

- Se Puede pensar en los fragmentos de sustantivo como un sustantivo más las palabras que describen el sustantivo, por ejemplo, en [la canción de Sheb Wooley de 1958] (https://en.wikipedia.org/wiki/The_Purple_People_Eater), un *"one-eyed, one-horned, flying, purple people-eater"* sería un fragmento de sustantivo largo.

In [ ]:
doc9 = nlp(u"Autonomous cars shift insurance liability toward manufacturers.")

for chunk in doc9.noun_chunks:
    print(chunk.text)

In [ ]:
doc10 = nlp(u"Red cars do not carry higher insurance rates.")

for chunk in doc10.noun_chunks:
    print(chunk.text)

In [ ]:
doc11 = nlp(u"He was a one-eyed, one-horned, flying, purple people-eater.")

for chunk in doc11.noun_chunks:
    print(chunk.text)


## 2. Stemming

### ¿Qué es el Stemming?

El **stemming** es un proceso heurístico que reduce una palabra a su **raíz o stem**, cortando sufijos mediante reglas predefinidas. El resultado **no es necesariamente una palabra válida**.

```
"corriendo"  →  "corr"
"corremos"   →  "corr"
"corredores" →  "corr"
```
### Historia y Origen

#### Martin F. Porter (1948 – presente)

El algoritmo de **Porter** es una técnica de Procesamiento de Lenguaje Natural (PLN) diseñada para reducir las palabras en inglés a su raíz o forma base (un proceso conocido como stemming) eliminando sufijos morfológicos. Su objetivo es agrupar palabras con significados similares para simplificar el análisis de textos

El **Algoritmo de Porter** fue creado por **Martin F. Porter**, matemático e informático británico, mientras trabajaba en el **Computer Laboratory de la Universidad de Cambridge**.

```
LÍNEA DEL TIEMPO DEL PORTER STEMMER
═══════════════════════════════════════════════════════════════════

  1960s                1980               1997              2006
    │                    │                  │                 │
    ▼                    ▼                  ▼                 ▼
 ┌──────────┐      ┌───────────┐     ┌──────────┐    ┌────────────┐
 │ Lovins   │      │  Porter   │     │ Snowball │    │  Porter2   │
 │ Stemmer  │─────▶│ Stemmer   │────▶│ lenguaje │───▶│ (English)  │
 │(1968,    │      │(publicado │     │  para    │    │  versión   │
 │ primera  │      │ 1980 en   │     │ stemmers │    │  mejorada  │
 │ propuesta│      │ Program   │     │          │    │            │
 │ formal)  │      │ journal)  │     │          │    │            │
 └──────────┘      └───────────┘     └──────────┘    └────────────┘
                        │
                   "An algorithm
                   for suffix
                   stripping"
                   Program, 14(3)
                   pp. 130-137

═══════════════════════════════════════════════════════════════════
```

#### El artículo original (1980)

El artículo seminal **"An algorithm for suffix stripping"** fue publicado en 1980 en la revista *Program: electronic library and information systems* (Vol. 14, No. 3, pp. 130-137).

| Dato | Detalle |
|---|---|
| **Autor** | Martin F. Porter |
| **Institución** | Computer Laboratory, Universidad de Cambridge |
| **Publicación** | *Program*, Vol. 14, No. 3, 1980 |
| **Objetivo** | Reducir variantes morfológicas a una forma raíz común |
| **Idioma** | Inglés (diseñado específicamente para inglés) |
| **Tipo** | Algorítmico (sin diccionario) |
| **Código** | Liberado como código abierto por el propio Porter |

#### Contexto histórico

En los años 1970s y 1980s, los primeros sistemas de **Recuperación de Información (IR)** enfrentaban un problema fundamental:

```
PROBLEMA:
  Una consulta con "running" no encontraba documentos con "run" o "runs".
  Una consulta con "connection" no encontraba "connecting" ni "connected".

SOLUCIÓN DE PORTER:
  Reducir TODAS las variantes a la misma raíz antes de indexar/buscar.

  running    ──▶  run
  runs       ──▶  run
  run        ──▶  run
  connected  ──▶  connect
  connection ──▶  connect
  connecting ──▶  connect
```

> ⚠️ **Nota importante:** Porter Stemmer **no** produce una raíz lingüística real (lema). Produce un *stem* que puede no ser una palabra válida del idioma. El objetivo es la consistencia, no la corrección lingüística.

#### Stemming vs Lematización

```
┌─────────────────────────────────────────────────────────────────┐
│                    STEMMING vs LEMATIZACIÓN                     │
├────────────────────────┬────────────────────────────────────────┤
│      STEMMING          │         LEMATIZACIÓN                   │
│   (Porter Stemmer)     │       (WordNet / spaCy)                │
├────────────────────────┼────────────────────────────────────────┤
│ "studies"  → "studi"   │ "studies"  → "study"                   │
│ "studying" → "studi"   │ "studying" → "study"                   │
│ "studied"  → "studi"   │ "studied"  → "study"                   │
│ "caresses" → "caress"  │ "caresses" → "caress"                  │
│ "dogs"     → "dog"     │ "dogs"     → "dog"                     │
│ "better"   → "better"❌│ "better"   → "good"   ✅              │
│ "running"  → "run"     │ "running"  → "run"                     │
├────────────────────────┼────────────────────────────────────────┤
│ ✅ Muy rápido           │ ✅ Preciso, produce palabras reales     │
│ ✅ Sin diccionario      │ ❌ Necesita diccionario/modelo          │
│ ❌ No siempre válido    │ ❌ Más lento                            │
│ ❌ Solo inglés          │ ✅ Multilingüe (con modelo adecuado)    │
└─────────────────────────┴──────────────────────────────────────────┘
```

#### Morphological Analysis (Análisis Morfológico)

```
Palabra: "connections"

┌──────────────────────────────────────────────────────────┐
│  c - o - n - n - e - c - t - i - o - n - s               │
│  └─────────────────────────────┘   └───┘ └┘              │
│              STEM                 SUFFIX INFLEXIÓN       │
│           "connect"              "ion"   "s"             │
│                                                          │
│  Porter produce: "connect"                               │
│  (elimina "-ions" y "-s" en pasos separados)             │
└──────────────────────────────────────────────────────────┘
```
#### Conceptos Clave: Medida *m* y Notación

##### Clasificación de letras

El algoritmo clasifica cada letra como **vocal (V)** o **consonante (C)**:

```
VOCALES:     a, e, i, o, u
CONSONANTES: todas las demás

REGLA ESPECIAL para "y":
  • Si "y" está precedida por una consonante → es VOCAL
  • En cualquier otro caso → es CONSONANTE

  Ejemplos:
  "by"   →  b(C) y(V)     ← y es vocal porque sigue a C
  "say"  →  s(C) a(V) y(C)← y es consonante porque sigue a V
  "yes"  →  y(C) e(V) s(C) ← y inicial es consonante
```

##### La representación de la forma

Porter usa la siguiente notación para representar la **estructura** de una palabra:

```
NOTACIÓN:
  C = secuencia de una o más consonantes
  V = secuencia de una o más vocales

  Forma general de un stem:
  [C](VC){m}[V]

  donde:
    [C] → 0 o 1 grupo de consonantes (opcional al inicio)
    (VC){m} → exactamente m repeticiones del patrón VC
    [V] → 0 o 1 grupo de vocales (opcional al final)
    m → la MEDIDA del stem
```

##### La Medida *m*

La **medida** es el número de pares Vocal–Consonante (VC) en el stem.

```
╔══════════════════════════════════════════════════════════════╗
║                    EJEMPLOS DE MEDIDA m                      ║
╠══════════╦═════════════════╦═════╦══════════════════════════╣
║ Palabra  ║ C/V Clasificada ║  m  ║ Explicación              ║
╠══════════╬═════════════════╬═════╬══════════════════════════╣
║ "tr"     ║ C  C            ║  0  ║ Solo consonantes          ║
║ "ee"     ║ V  V            ║  0  ║ Solo vocales              ║
║ "tree"   ║ C  C  V  V      ║  0  ║ [CC][VV] → m=0           ║
║ "trouble"║ C C V C C V V   ║  1  ║ [CC][V][CC][VV] → m=1   ║
║ "oaten"  ║ V V C V C       ║  1  ║ [VV][C][V][C] → m=1     ║
║ "trees"  ║ C C V V C       ║  1  ║ [CC][VV][C] → m=1       ║
║ "ivy"    ║ V C V           ║  1  ║ [V][C][V] → m=1         ║
║ "orrery" ║ V C C V C V     ║  2  ║ [V][CC][V][C][V] → m=2  ║
║ "genera" ║ C V C V C V     ║  2  ║ [C][V][C][V][C][V] →m=2 ║
╚══════════╩═════════════════╩═════╩══════════════════════════╝
```

> 💡 **¿Por qué importa m?** Las reglas del algoritmo solo se aplican cuando el stem resultante tiene `m > 0` o `m > 1`. Esto evita sobre-reducir palabras cortas.
>
> Ejemplo: "caressed" tiene m=2 → se aplica la regla `ed →`
> Ejemplo: "bed" tiene m=1 → NO se aplica `ed →` (el stem resultante sería "b", m=0)

### Los 5 Pasos del Algoritmo

### ¿Cómo funciona el Porter Stemmer?

Aplica reglas en 5 pasos secuenciales:

```
Paso 1a: sses → ss, ies → i, ss → ss, s → ""
Paso 1b: eed → ee, ed → "", ing → ""
Paso 1c: y → i
Paso 2:  ational → ate, tional → tion, ...
Paso 3:  icate → ic, ative → "", alize → al
Paso 4:  al, ance, ence, er, ic, ... → ""
Paso 5a: e → "" (condicional)
Paso 5b: ll → l (condicional)
```
[Porter Stemmer Algorithm](https://developer.ibm.com/tutorials/awb-stemming-text-porter-stemmer-algorithm-python/)

```
FLUJO GENERAL DEL PORTER STEMMER
════════════════════════════════════════════════════════════

  Palabra de entrada: "GENERALIZATION"
         │
         ▼
  ┌─────────────┐
  │   PASO 1a   │  Plurales e inflexiones básicas
  │  sses/ies/s │  "caresses" → "caress"
  └──────┬──────┘
         │
         ▼
  ┌─────────────┐
  │   PASO 1b   │  Pasado y progresivo (-ed, -ing)
  │  eed/ed/ing │  "motoring" → "motor"
  └──────┬──────┘
         │
         ▼
  ┌─────────────┐
  │   PASO 1c   │  -y → -i
  │     y→i     │  "happy" → "happi"
  └──────┬──────┘
         │
         ▼
  ┌─────────────┐
  │   PASO 2    │  Sufijos largos dobles
  │ ational/     │  "relational" → "relate"
  │ tional/etc  │  "generalization" → "generalize"
  └──────┬──────┘
         │
         ▼
  ┌─────────────┐
  │   PASO 3    │  Sufijos de adjetivos
  │ icate/alize │  "generalize" → "general"
  │ ness/ful    │  "hopeful" → "hope"
  └──────┬──────┘
         │
         ▼
  ┌─────────────┐
  │   PASO 4    │  Sufijos nominales (m > 1)
  │ al/ance/er/ │  "general" → "gener"
  │ ism/ate/ize │
  └──────┬──────┘
         │
         ▼
  ┌─────────────┐
  │   PASO 5a   │  -e final
  │  e → (vacío)│  "probate" → "probat"
  └──────┬──────┘
         │
         ▼
  ┌─────────────┐
  │   PASO 5b   │  -ll → -l
  │ ll → l      │  "controll" → "control"
  └──────┬──────┘
         │
         ▼
  Stem resultante: "GENERAL" → "gener"

════════════════════════════════════════════════════════════
```

#### Paso 1a — Plurales básicos

| Condición | Sufijo | Reemplaza con | Ejemplo |
|---|---|---|---|
| — | `sses` | `ss` | caresses → caress |
| — | `ies` | `i` | ponies → poni |
| — | `ss` | `ss` | caress → caress |
| — | `s` | `` | cats → cat |

#### Paso 1b — Pasado y progresivo

| Condición | Sufijo | Reemplaza con | Ejemplo |
|---|---|---|---|
| (m > 0) | `eed` | `ee` | agreed → agree |
| (*v* presente en stem) | `ed` | `` | plastered → plaster |
| (*v* presente en stem) | `ing` | `` | motoring → motor |

**Reglas adicionales** si el paso 1b transformó el stem con `ed` o `ing`:

```
  Si el stem termina en: at, bl, iz  →  agregar 'e'
    conflat(ed) → conflate
    troubl(ing) → trouble
    siz(ed)     → size

  Si el stem termina en consonante doble
  y NO termina en l, s, z             →  quitar la última letra
    hopp(ing)   → hop
    tann(ed)    → tan
    fall(ing)   → fall  (no se aplica, termina en l)

  Si (m=1) y el stem termina en CVC  →  agregar 'e'
  (la última C no puede ser w, x, y)
    hop → hope  (m=1, termina en h-o-p)
    mat → mate  (m=1, termina en m-a-t)
```

#### Paso 1c — Y final

| Condición | Sufijo | Reemplaza | Ejemplo |
|---|---|---|---|
| (*v* presente en stem) | `y` | `i` | happy → happi, sky → sky |

#### Paso 2 — Sufijos dobles (m > 0)

| Sufijo | Reemplaza con | Ejemplo |
|---|---|---|
| `ational` | `ate` | relational → relate |
| `tional` | `tion` | conditional → condition |
| `enci` | `ence` | valenci → valence |
| `anci` | `ance` | hesitanci → hesitance |
| `izer` | `ize` | digitizer → digitize |
| `abli` | `able` | conformabli → conformable |
| `alli` | `al` | radicalli → radical |
| `entli` | `ent` | differentli → different |
| `eli` | `e` | vileli → vile |
| `ousli` | `ous` | analogousli → analogous |
| `ization` | `ize` | digitization → digitize |
| `ation` | `ate` | predication → predicate |
| `ator` | `ate` | operator → operate |
| `alism` | `al` | feudalism → feudal |
| `iveness` | `ive` | decisiveness → decisive |
| `fulness` | `ful` | hopefulness → hopeful |
| `ousness` | `ous` | callousness → callous |
| `aliti` | `al` | formaliti → formal |
| `iviti` | `ive` | sensitiviti → sensitive |
| `biliti` | `ble` | sensibiliti → sensible |

#### Paso 3 — Sufijos (m > 0)

| Sufijo | Reemplaza con | Ejemplo |
|---|---|---|
| `icate` | `ic` | triplicate → triplic |
| `ative` | `` | formative → form |
| `alize` | `al` | formalize → formal |
| `iciti` | `ic` | electriciti → electric |
| `ical` | `ic` | electrical → electric |
| `ful` | `` | hopeful → hope |
| `ness` | `` | goodness → good |

#### Paso 4 — Sufijos adicionales (m > 1)

| Sufijo | Ejemplo |
|---|---|
| `al` | revival → reviv |
| `ance` | allowance → allow |
| `ence` | inference → infer |
| `er` | airliner → airlin |
| `ic` | electric → electr |
| `able` | adjustable → adjust |
| `ible` | defensible → defens |
| `ant` | irritant → irrit |
| `ement` | replacement → replac |
| `ment` | adjustment → adjust |
| `ent` | dependent → depend |
| `ion` (si stem en s/t) | adoption → adopt |
| `ou` | homologou → homolog |
| `ism` | communism → commun |
| `ate` | activate → activ |
| `iti` | angulariti → angular |
| `ous` | homologous → homolog |
| `ive` | effective → effect |
| `ize` | bowdlerize → bowdler |

#### Pasos 5a y 5b — Limpieza final

```
PASO 5a:
  • (m > 1)  'e' final → eliminar        "probate" → "probat"
  • (m = 1)  'e' final → eliminar        "cease"   → "ceas"
    (solo si el stem no termina en CVC)
    Excepción: (m=1) 'e' → no eliminar   "rate"    → "rate"  (m=1, *o = true)

PASO 5b:
  • (m > 1) y stem termina en 'l' doble → eliminar última 'l'
    "controll" → "control"
    "roll"     → "roll"  (m=1, no aplica)
```

Limitaciones y Alternativas <a id='limitaciones'></a>

### Casos donde el Porter Stemmer falla

```
OVER-STEMMING (sobre-reducción):
  Palabras distintas → mismo stem incorrecto

  "news"     → "new"      ❌ (no está relacionado)
  "universe" → "univers"  ← stem no es palabra
  "data"     → "data"     (no cambia, correcto)
  "general"  → "gener"    ← stem no es palabra válida

UNDER-STEMMING (sub-reducción):
  Palabras relacionadas → stems distintos

  "alumnus"  → "alumnu"   )
  "alumni"   → "alumni"   )  stems distintos ❌

  "better"   → "better"   )  no encuentra raíz
  "good"     → "good"     )  en "good" ❌
```

#### Comparativa de stemmers

| Stemmer | Año | Agresividad | Errores OE | Errores UE | Velocidad |
|---|---|---|---|---|---|
| **Porter** | 1980 | Media | Bajo | Medio | ✅✅✅ |
| **Porter2 / Snowball** | 2006 | Media-alta | Bajo | Bajo | ✅✅✅ |
| **Lancaster** | 1990 | Alta | **Alto** | Bajo | ✅✅ |
| **Lovins** | 1968 | Alta | Alto | Bajo | ✅✅ |
| **Snowball ES** | 2002 | Media | Bajo | Medio | ✅✅✅ |

> **OE** = Over-stemming (sobre-reducción) · **UE** = Under-stemming (sub-reducción)

#### ¿Cuándo usar Porter Stemmer?

```
USA Porter Stemmer cuando:
  ✅ Trabajas con inglés
  ✅ Necesitas velocidad (big data, IR a gran escala)
  ✅ La tarea no requiere palabras válidas (BoW, TF-IDF)
  ✅ El corpus es muy grande
  ✅ Quieres un baseline rápido

USA Lematización en su lugar cuando:
  ✅ Necesitas palabras gramaticalmente válidas
  ✅ Trabajas con textos para humanos
  ✅ Tienes corpus pequeños o medianos
  ✅ La tarea requiere interpretabilidad
  ✅ Trabajas con idiomas con morfología compleja (español, alemán)
```

### Stemmers para Español <a id='espanol'></a>

#### ¿Por qué el español es más difícil?

El español tiene una **morfología mucho más rica** que el inglés. Un mismo verbo puede tener decenas de formas conjugadas, y los sustantivos/adjetivos varían en género y número:

```
INGLÉS — verbo "to speak" (7 formas):
  speak, speaks, spoke, spoken, speaking, speaker, speakers

ESPAÑOL — verbo "hablar" (+50 formas):
  hablar, hablo, hablas, habla, hablamos, habláis, hablan,
  hablé, hablaste, habló, hablamos, hablasteis, hablaron,
  hablaba, hablabas, hablaban, hablaré, hablarás, hablará,
  hablando, hablado, hablada, hablados, habladas,
  hablador, habladora, habladores, habladoras, ...
```

Además, el español tiene:
- **Género gramatical**: niño/niña, doctor/doctora, alumno/alumna
- **Acento ortográfico**: canción → canciones (pierde el acento), árbol → árboles
- **Diptongos en conjugación**: poder → puedo (no *podo*)
- **Verbos irregulares**: ser, ir, tener, hacer, decir, venir, poner...

#### Algoritmos disponibles para español

```
╔══════════════════════════════════════════════════════════════════════════╗
║            STEMMERS PARA ESPAÑOL — RESUMEN COMPARATIVO                   ║
╠══════════════╦═══════════╦════════════╦══════════════╦══════════════════╣
║ Algoritmo    ║   Año     ║  Tipo      ║  Disponible  ║  Características ║
╠══════════════╬═══════════╬════════════╬══════════════╬══════════════════╣
║ Savoy        ║  1999     ║  Light     ║  Investigación║ Conservador,    ║
║              ║           ║  stemming  ║               ║ 29 reglas       ║
╠══════════════╬═══════════╬════════════╬══════════════╬══════════════════╣
║ Snowball ES  ║  2002     ║  Reglas    ║  NLTK/       ║ Más completo,   ║
║ (Porter ES)  ║           ║  sufijos   ║  PyStemmer   ║ maneja acentos  ║
╠══════════════╬═══════════╬════════════╬══════════════╬══════════════════╣
║ Light        ║  2003     ║  Mínimo    ║  Fácil impl. ║ Solo plurales y ║
║ Stemmer ES   ║           ║            ║               ║ sufijos simples ║
╠══════════════╬═══════════╬════════════╬══════════════╬══════════════════╣
║ S-Stemmer    ║  2004     ║  Sufijos   ║  Investigación║ Corpus CESS-ESP ║
║              ║           ║            ║               ║                 ║
╠══════════════╬═══════════╬════════════╬══════════════╬══════════════════╣
║ spaCy ES     ║  2017+    ║  Lematiza- ║  pip install ║ Lematizador,   ║
║ (no stemmer) ║           ║  ción real ║  spacy        ║ no stemmer      ║
╚══════════════╩═══════════╩════════════╩══════════════╩══════════════════╝
```

---

#### Snowball para Español (el más usado)

**Snowball** es un framework creado por el propio **Martin Porter** (2001) para escribir algoritmos de stemming para múltiples idiomas. El stemmer de español fue escrito siguiendo la misma lógica que Porter, pero adaptado a la morfología española.

```
ESTRUCTURA DEL SNOWBALL ESPAÑOL:

  1. Normalización de acentos: á→a, é→e, í→i, ó→o, ú→u  (en el stem)
  2. Encontrar la región R1 y R2 (igual que Porter)
  3. Eliminar sufijos estándar (nominales):
       -anza, -anzas, -ico, -ica, -icos, -icas
       -ismo, -ismos, -able, -ables, -ible, -ibles
       -ista, -istas, -oso, -osa, -osos, -osas
       -amiento, -amientos, -imiento, -imientos
  4. Eliminar sufijos verbales:
       -ando, -iendo, -ar, -er, -ir
       -aré, -arás, -ará, -aremos, -aréis, -arán
       -ería, -erías, -eríamos, -ieron, -ierais
       ... (más de 90 terminaciones verbales)
  5. Eliminar sufijos residuales:
       -os, -a, -as, -e, -es, -en, -ó
  6. Quitar acentos del stem final
```

##### Regiones R1 y R2

```
R1 = subcadena después del primer par no-vocal + vocal
R2 = R1 aplicado de nuevo a R1

Ejemplo: "procesamiento"
  p r o c e s a m i e n t o
  └─────────┘
  R1 comienza en la 'a' de "cesa"
  → R1 = "amiento"
  → R2 = dentro de R1 = "miento"
```

### Otros Algoritmos de Stemming

| Algoritmo | Idioma | Descripción |
|---|---|---|
| **Porter** | Inglés | El más conocido (1980), 5 fases de reducción |
| **Snowball** | 15+ idiomas | Más preciso que Porter, incluye español |
| **Lancaster** | Inglés | Muy agresivo, produce raíces muy cortas |
| **ISRI** | Árabe | Para texto árabe |


#### Ejemplo Simple de Stemmer 

In [ ]:
import unicodedata
help(unicodedata)

[Unicode Normalization](https://theproductguy.in/blogs/unicode-normalization-python/)

In [ ]:
#Normalizando el texto para quitar acentos
def sin_acento(s):
        return ''.join(c for c in unicodedata.normalize('NFD', s)
                       if unicodedata.category(c) != 'Mn')


In [ ]:
#creando una función simple para simular el algoruitmo de stemmer en español
def stemmer_simple_es(palabra):
    """Stemmer simplificado para espanol (sufijos comunes ordenados por longitud)."""
    
    #Proponemos algunos sufijos
    sufijos = [
        'imientos', 'amiento', 'aciones', 'izacion', 'adores',
        'amente', 'iendo', 'acion', 'mente', 'ando',
        'ados', 'adas', 'idos', 'idas', 'ores',
        'amos', 'emos', 'imos', 'cion', 'dad',
        'ar', 'er', 'ir', 'os', 'as', 'es',
    ]
    #Proceso de normalización
    

    #Ordenando los sufijos
    
    return p


In [ ]:
palabras_prueba = [
    'corriendo', 'comiendo', 'clasificacion', 'rapidamente',
    'procesadores', 'aprendidas', 'estudiamos', 'tokenizar'
]

print('Stemmer simple para espanol (reglas de sufijos):')
print(f'  {"Palabra":<25} {"Stem"}')
print('  ' + '-'*40)
for p in palabras_prueba:
    print(f'  {p:<25} -> {stemmer_simple_es(p)}')

### Stemming con NLTK <a id='nltk-stem'></a>

In [ ]:
from nltk.stem import PorterStemmer, SnowballStemmer, LancasterStemmer

# Instanciar los tres stemmers para ingles
porter      = PorterStemmer()
snowball_en = SnowballStemmer('english')
lancaster   = LancasterStemmer()

# Ingles: comparacion de algoritmos
palabras_en = ['running', 'easily', 'fairly', 'generously', 'processing',
               'studies', 'beautiful', 'connection', 'happiness']

In [ ]:
print('Comparacion de Stemmers (ingles):')
print(f'  {"Palabra":<15} {"Porter":<12} {"Snowball":<12} {"Lancaster"}')
print('  ' + '-'*55)
for w in palabras_en:
    print(f'  {w:<15} {porter.stem(w):<12} {snowball_en.stem(w):<12} {lancaster.stem(w)}')

In [ ]:
# Crear stemmer Snowball para espanol
snowball_es = SnowballStemmer('spanish')

# Espanol: SnowballStemmer
palabras_es = [
    'procesamiento', 'procesando', 'procesados', 'procesador',
    'tokenizacion', 'tokenizando', 'aprendizaje', 'aprendiendo',
    'clasificacion', 'clasificador', 'estudiantes', 'estudiar'
]

In [ ]:
print('Snowball Stemmer (español):')
print(f'  {"Palabra":<20} {"Stem"}')
print('  ' + '-'*35)
for p in palabras_es:
    print(f'  {p:<20} -> {snowball_es.stem(p)}')

##### Ejemplo de snowball para un corpus

In [ ]:
import matplotlib.pyplot as plt

corpus = [
    "El aprendizaje automático requiere muchos datos",
    "Aprender machine learning es fascinante",
    "Los datos son fundamentales en el aprendizaje",
    "Procesamiento de datos con Python",
    "El procesador de lenguaje natural procesa textos",
    "Clasificación automática de documentos",
    "Clasificar textos con modelos entrenados",
]


In [ ]:
#Tokenización


In [ ]:
# APlicando snowball para todos los tokens


In [ ]:
# # Visualización: reducción de vocabulario con stemming 
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(['Vocabulario\noriginal', 'Vocabulario\ncon stemming'],
              [len(vocab_original), len(vocab_stemmed)],
              color=['steelblue', 'tomato'], width=0.5)
ax.bar_label(bars, fontsize=12, fontweight='bold')
ax.set_title('Reducción de vocabulario con Stemming', fontsize=12)
ax.set_ylabel('Número de palabras únicas')
plt.tight_layout()
plt.show()

## 3. Lematización

### ¿Qué es la Lematización?

La **lematización** reduce una palabra a su **forma base o lema** usando análisis morfológico y diccionarios. A diferencia del stemming, el lema **siempre es una palabra válida**.

```
"corriendo"  →  "correr"    ✅ (palabra real)
"corredores" →  "corredor"  ✅
"mejores"    →  "bueno"     ✅ (supletivismo)
```

### Stemming vs Lematización: Diferencia clave

| Característica | Stemming | Lematización |
|---|---|---|
| Resultado | Raíz (puede no ser palabra) | Lema (siempre es palabra) |
| Velocidad | Muy rápido (reglas) | Más lento (diccionario) |
| Precisión | Menor | Mayor |
| Requiere POS | No | Sí (para desambiguar) |
| `better` | `better` (Porter) | `good` (lematización) |
| `running` | `run` | `run` |

### El problema de la desambiguación

```
"saw"  →  como VERBO: "see"
"saw"  →  como SUSTANTIVO: "saw" (una sierra)
```
Por eso la lematización necesita el **Part-of-Speech (POS)** del token.

#### Ejemplo de Lematización Simple

In [ ]:
# Diccionario con Lemas en español
LEMAS_ES = {
    # Verbos irregulares
    "fui": "ir", "fue": "ir", "voy": "ir", "va": "ir", "van": "ir",
    "soy": "ser", "es": "ser", "era": "ser", "son": "ser", "fue": "ser",
    "tengo": "tener", "tiene": "tener", "tuvo": "tener", "tuvimos": "tener",
    # Sustantivos
    "niños": "niño", "niñas": "niña", "gatos": "gato", "perros": "perro",
    # Adjetivos
    "mejores": "bueno", "peores": "malo", "mayores": "grande",
    # Formas verbales
    "corriendo": "correr", "comiendo": "comer", "aprendiendo": "aprender",
    "corría": "correr", "comía": "comer",
}

def lematizar_manual(palabra):
    return LEMAS_ES.get(palabra.lower(), palabra.lower())

palabras = ["niños", "mejores", "fui", "corriendo", "aprendiendo", "tengo", "mayores"]

print("Lematización manual:")
for p in palabras:
    print(f"  {p:<20} → {lematizar_manual(p)}")

### Lematización con NLTK y spaCy

In [ ]:
from nltk.stem import WordNetLemmatizer

In [ ]:
# Lematización con NLTK 
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

`wordnet`: Descarga la base de datos WordNet de la Universidad de Princeton, un diccionario léxico gigante para el idioma inglés. Agrupa sustantivos, verbos, adjetivos y adverbios en conjuntos de sinónimos cognitivos (llamados synsets). Se usa principalmente para la lematización (reducir una palabra a su forma raíz).

`omw-1.4`: Descarga los datos de Open Multilingual Wordnet (WordNet Multilingüe Abierto). 

Las versiones modernas de NLTK requieren este paquete junto con el WordNet estándar para vincular palabras en varios idiomas o leer diccionarios estructurales específicos.

`quiet=True`: Elimina las alertas visuales estándar del tipo [nltk_data] Downloading package.... Es muy útil si ejecutas scripts en entornos de producción automatizados.

In [ ]:
#instanciando el objeto para lematizar
wnl = WordNetLemmatizer()

# Palabras de prueba en inglés (palabra y el pos)
palabras_en = [
    ("running", "v"), ("better", "a"), ("geese", "n"),
    ("went", "v"), ("wolves", "n"), ("happily", "r"),
    ("studies", "v"), ("studies", "n"), ("saw", "v"), ("saw", "n")
]

In [ ]:
print("WordNetLemmatizer (NLTK) — inglés:")
print(f"  {'Palabra':<15} {'POS':<6} {'Lema'}")
print("  " + "-"*35)
for palabra, pos in palabras_en:
    lema = wnl.lemmatize(palabra, pos=pos)
    pos_name = {'v':'VERBO','n':'SUST.','a':'ADJ.','r':'ADV.'}[pos]
    print(f"  {palabra:<15} {pos_name:<6} → {lema}")

#### Lematización Con Pos (NLTK + pos_tag)

In [ ]:
from nltk import pos_tag
from nltk.corpus import wordnet
#generando los pos
def get_wn_pos(tag):
    if tag.startswith('J'): return wordnet.ADJ
    elif tag.startswith('V'): return wordnet.VERB
    elif tag.startswith('R'): return wordnet.ADV
    else: return wordnet.NOUN

#Obteniendo la lematzación con pos
def lematizar(texto, idioma='english'):
    
    return


In [ ]:
frases = [
    'The researchers were running experiments on better computing systems.',
    'The mice went to the children teeth doctor.',
    'Studies have shown that going faster is not always better.',
]

for frase in frases:
    r = lematizar(frase) # obtiene los tokens y los lemas
    print(f'Frase: {frase}')
    print(f'  {"Token":<20} {"POS":<8} {"Lema"}')
    print('  '+'-'*45)
    for t,pos,lema in r:
        if len(t)>1: print(f'  {t:<20} {pos:<8} {lema}')
    print()

In [ ]:
#  Lematización con spaCy (español) 
textos = [
    "Los estudiantes estaban aprendiendo nuevas técnicas de programación.",
    "Los mejores modelos fueron entrenados con millones de documentos.",
    "Las redes neuronales profundas han transformado el procesamiento."
]

print("spaCy para Lematización en ESPAÑOL:")
print("=" * 60)

for texto in textos:
    doc = nlp_es(texto)
    print(f"\nTexto: {texto}")
    print(f"  {'Token':<20} {'POS':<8} {'Lema'}")
    print("  " + "-"*45)
    for token in doc:
        if not token.is_punct and not token.is_space:
            print(f"  {token.text:<20} {token.pos_:<8} {token.lemma_}")

## 4. Comparativa: Stemming vs Lematización

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Comparación directa 
palabras_comp = [
    "running", "better", "geese", "fairly", "generously",
    "connection", "hapiness", "wolves", "studies", "organization"
]

rows = []
for p in palabras_comp:
    rows.append({
        'Palabra': p,
        'Porter': porter.stem(p),
        'Snowball': snowball_en.stem(p),
        'WordNet (n)': wnl.lemmatize(p, 'n'),
        'WordNet (v)': wnl.lemmatize(p, 'v'),
    })

df = pd.DataFrame(rows)
print("COMPARATIVA COMPLETA: Stemming vs Lematización")
print(df.to_string(index=False))